# Huấn luyện QA-Graph cho VG và PSG trên Colab GPU



In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, importlib.util, json, os, subprocess, sys
from google.colab import drive
drive.mount('/content/drive')
DRIVE = Path('/content/drive/MyDrive')
REPO = DRIVE / 'graphml-project3'
EXP = REPO / 'src/mmxai/xai/xai-experiment'
sys.path[:0] = [str(EXP), str(REPO / 'src')]
from xai_experiment.paths import runs_dir
RUNS = runs_dir()
SEEDS = (0, 1, 2)
RUN_LABEL = datetime.now(timezone.utc).strftime('dual-%Y%m%d-%H%M%S')
assert (REPO / 'src/mmxai/graph/qa.py').is_file()
assert (EXP / 'xai_experiment/model.py').is_file(), 'Đồng bộ repo mới lên Drive trước khi chạy.'
if importlib.util.find_spec('torch_geometric') is None or importlib.util.find_spec('yaml') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO)], check=True)
import torch
assert torch.cuda.is_available(), 'Chọn GPU trong Colab Runtime.'
from xai_experiment.model import MODEL_SPECS
RUNS.mkdir(parents=True, exist_ok=True)
print('GPU:', torch.cuda.get_device_name(0), '| run:', RUN_LABEL)


In [ ]:
from mmxai.data.bundle import resolve_extra
BUNDLES = {name: DRIVE / spec.bundle_dir for name, spec in MODEL_SPECS.items()}
BASE_FILES = ('data/images.jsonl', 'data/captions.jsonl', 'data/object_vocab_train.json',
              'features/meta.json', 'split/split.json')
def sha256(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(4 * 1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

SOURCE_FILES = ('src/mmxai/graph/qa.py', 'src/mmxai/graph/build.py',
                'src/mmxai/data/bundle.py', 'src/mmxai/experiments/vg_qa.py',
                'src/mmxai/experiments/vg_graph.py', 'src/mmxai_eval.py',
                'src/mmxai/xai/xai-experiment/xai_experiment/model.py',
                'src/mmxai/xai/xai-experiment/xai_experiment/perturbation.py',
                'src/mmxai/xai/xai-experiment/xai_experiment/explain.py',
                'src/mmxai/xai/xai-experiment/xai_experiment/evaluate_xai.py',
                'src/mmxai/xai/xai-experiment/xai_experiment/scorer_adapter.py',
                'src/mmxai/xai/xai-experiment/xai_experiment/graph.py',
                'src/mmxai/xai/xai-experiment/xai_experiment/io.py')
source_fingerprints = {rel: sha256(REPO / rel) for rel in SOURCE_FILES}
fingerprints, bundle_extras = {}, {}
for name, spec in MODEL_SPECS.items():
    bundle = BUNDLES[name]
    extra = resolve_extra(bundle, 'graph_extra_cn')
    required = [bundle / rel for rel in BASE_FILES]
    features = sorted((bundle / 'features').glob('clip_*.npz'))
    assert features, f'{name}: missing CLIP features in {bundle}'
    required += [features[0], extra / 'labels.json', extra / 'label_emb.npy', extra / 'conceptnet.tsv']
    missing = [str(path) for path in required if not path.is_file()]
    assert not missing, f'{name}: thiếu {missing}'
    bundle_extras[name] = extra
    optional_eval = bundle / 'mmxai_eval.py'
    if optional_eval.is_file():
        required.append(optional_eval)
    meta = json.loads((bundle / 'features/meta.json').read_text())
    labels = json.loads((extra / 'labels.json').read_text())
    edges = sum(1 for _ in (extra / 'conceptnet.tsv').open())
    assert meta['model'] == 'ViT-B-32' and meta['pretrained'] == 'laion2b_s34b_b79k'
    assert len(labels) == spec.expected_labels and 2 * edges == spec.expected_concept_edges
    fingerprints[name] = {str(path.relative_to(bundle)): sha256(path) for path in required}
    print(name, '|', bundle, '| labels:', len(labels), '| directed CN edges:', 2 * edges)


In [ ]:
run_records = []
for name, spec in MODEL_SPECS.items():
    for seed in SEEDS:
        run_id = f'{RUN_LABEL}_{name}_seed{seed}'
        run_dir = RUNS / spec.experiment / run_id
        assert not run_dir.exists(), f'Run đã tồn tại: {run_dir}'
        cmd = [sys.executable, '-m', 'mmxai.run', '--exp', spec.experiment,
               '--config-dir', str(REPO / 'configs'), '--seed', str(seed),
               '--set', f'data={BUNDLES[name]}',
               '--set', f'extra={bundle_extras[name]}',
               '--set', 'save_model=true', '--runs-dir', str(RUNS),
               '--run-id', run_id, '--no-publish']
        env = os.environ.copy()
        env['LAB_USER'] = 'duong'
        env['PYTHONHASHSEED'] = str(seed)
        env['PYTHONPATH'] = str(REPO / 'src') + os.pathsep + env.get('PYTHONPATH', '')
        print('Running', name, 'seed', seed, flush=True)
        subprocess.run(cmd, cwd=REPO, env=env, check=True)
        checkpoint = run_dir / 'model.pt'
        assert checkpoint.is_file() and checkpoint.stat().st_size > 0
        metrics = json.loads((run_dir / 'metrics.json').read_text())
        assert metrics['run']['error'] is None, metrics['run']['error']
        cfg = json.loads((run_dir / 'config.json').read_text())
        stats = metrics['run']['extra']['graph']
        assert cfg['graph']['variant'] == spec.graph_variant and cfg['gnn']['kind'] == spec.gnn_kind
        assert stats['n_label'] == spec.expected_labels and stats['E_concept'] == spec.expected_concept_edges
        info = json.loads((run_dir / 'train_info.json').read_text())[0]
        record = {'dataset': name, 'experiment': spec.experiment, 'seed': seed,
                  'run_dir': str(run_dir), 'checkpoint_sha256': sha256(checkpoint),
                  'alpha': info['alpha'], 'test_full_MRR': metrics['metrics']['test_full']['MRR']['mean']}
        run_records.append(record)
        print('Saved', checkpoint, '| MRR', record['test_full_MRR'], '| alpha', record['alpha'])

manifest = {'created_at_utc': datetime.now(timezone.utc).isoformat(), 'run_label': RUN_LABEL,
            'bundle_sha256': fingerprints, 'source_sha256': source_fingerprints, 'runs': run_records}
manifest_path = RUNS / f'{RUN_LABEL}_manifest.json'
manifest_path.write_text(json.dumps(manifest, ensure_ascii=False, indent=2))
print('Manifest:', manifest_path)


## Sau khi huấn luyện

Kiểm tra manifest có đủ sáu lượt và `model.pt` của từng lượt. Sau đó chạy `02_baseline_xai_colab.ipynb`, notebook này sẽ nạp đúng bundle, config, checkpoint và α theo manifest. Dữ liệu test chỉ dùng để báo cáo và chọn case study cố định; checkpoint không được chọn theo MRR test.
